In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [48]:
!pip install transformers torch scikit-learn

In [49]:
df_review = pd.read_csv("B2W-Reviews01.csv")
# print(df_review.info())


/tmp/ipykernel_1875/2121650720.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_review = pd.read_csv("B2W-Reviews01.csv")


# Tratamento dos dados

In [50]:
#Tinhamos 18 valores que estavam nan no recommend_to_a_friend, vamos remover
df_review = df_review.dropna(subset=["recommend_to_a_friend"])

#Juntando as review_title + review_text, ja tirando as nan
df_review["review_completo"] = df_review["review_title"].fillna("") + " " + df_review["review_text"].fillna("")

#Cortando onde o review_completo foi feito de review_title vazio e df_review_text tmb vazio, sobrando só o espaço da juncao dos dois
df_review = df_review[df_review["review_completo"].str.strip().str.len() > 0]
# print(len(df_review))

print(df_review["review_completo"].str.split().str.len().describe())
print((df_review["review_completo"].str.split().str.len()> 64).sum())

df_review["num_palavras"] = df_review["review_completo"].str.split().str.len()
print(df_review["num_palavras"].describe())
print("\nTextos com mais de 64 palavras:", (df_review["num_palavras"] > 64).sum())
print("Textos com mais de 128 palavras:", (df_review["num_palavras"] > 128).sum())

count    132283.000000
mean         25.529758
std          22.687033
min           1.000000
25%          13.000000
50%          19.000000
75%          30.000000
max         802.000000
Name: review_completo, dtype: float64
6756
count    132283.000000
mean         25.529758
std          22.687033
min           1.000000
25%          13.000000
50%          19.000000
75%          30.000000
max         802.000000
Name: num_palavras, dtype: float64

Textos com mais de 64 palavras: 6756
Textos com mais de 128 palavras: 927


# Dividir entre treino, validacao e teste

In [51]:
X = df_review["review_completo"]
y = df_review["recommend_to_a_friend"]
y = y.map({"Yes":1, "No":0})
# print(X.head())
# print(y.unique())

X_treino, X_resto, y_treino, y_resto =train_test_split(X,y, train_size=0.6, stratify =y, random_state= 7)
X_validacao, X_teste, y_validacao, y_teste = train_test_split(X_resto,y_resto, test_size=0.5, stratify=y_resto, random_state= 7)


# Modelo DistilBert

In [52]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch


#Usar a gpu do collab se tiver disponivel
if torch.cuda.is_available():
    placa = torch.device("cuda")
else:
    placa = torch.device("cpu")
print("Tá usando: ", placa)

#Carregando tokenizador e o modelo
tokenizador = DistilBertTokenizer.from_pretrained("adalbertojunior/distilbert-portuguese-cased")
modelo_distilBert = DistilBertModel.from_pretrained("adalbertojunior/distilbert-portuguese-cased")
modelo_distilBert = modelo_distilBert.to(placa)
modelo_distilBert.eval()

#Extrair os embeddings
def extrair_embeddings(texts, batch_size=32):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        encoded = tokenizador(batch,max_length=128,truncation=True,padding="longest",return_tensors="pt").to(placa)

        with torch.no_grad():
            outputs = modelo_distilBert(**encoded)

        # Pega o token [CLS] (representação geral do texto)
        cls = outputs.last_hidden_state[:,0,:]
        cls = cls.cpu()
        cls = cls.numpy()
        embeddings.append(cls)


        print(f"  Processados {i}/{len(texts)}")

    return np.vstack(embeddings)

print("Extraindo embeddings de treino...")
X_treino_emb    = extrair_embeddings(X_treino.tolist())

print("Extraindo embeddings de validação...")
X_validacao_emb = extrair_embeddings(X_validacao.tolist())

print("Extraindo embeddings de teste...")
X_teste_emb     = extrair_embeddings(X_teste.tolist())

Tá usando:  cuda


[transformers] You are using a model of type `bert` to instantiate a model of type `distilbert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: adalbertojunior/distilbert-portuguese-cased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
encoder.layer.{0, 1, 2, 3, 4, 5}.attention.self.value.weight       | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.output.LayerNorm.bias             | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.output.dense.weight               | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.attention.output.LayerNorm.bias   | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.intermediate.dense.weight         | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.attention.self.query.weight       | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.attention.self.key.bias           | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.attention.output.LayerNorm.weight | UNEXPECTED | 
encoder.layer.{0, 1, 2, 3, 4, 5}.attention.self.key.weight         | UNEXPECT

Extraindo embeddings de treino...
  Processados 0/79369
  Processados 32/79369
  Processados 64/79369
  Processados 96/79369
  Processados 128/79369
  Processados 160/79369
  Processados 192/79369
  Processados 224/79369
  Processados 256/79369
  Processados 288/79369
  Processados 320/79369
  Processados 352/79369
  Processados 384/79369
  Processados 416/79369
  Processados 448/79369
  Processados 480/79369
  Processados 512/79369
  Processados 544/79369
  Processados 576/79369
  Processados 608/79369
  Processados 640/79369
  Processados 672/79369
  Processados 704/79369
  Processados 736/79369
  Processados 768/79369
  Processados 800/79369
  Processados 832/79369
  Processados 864/79369
  Processados 896/79369
  Processados 928/79369
  Processados 960/79369
  Processados 992/79369
  Processados 1024/79369
  Processados 1056/79369
  Processados 1088/79369
  Processados 1120/79369
  Processados 1152/79369
  Processados 1184/79369
  Processados 1216/79369
  Processados 1248/79369
  P

# Regressão Logistica

In [54]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

for i in[0.001, 0.01, 0.1, 1, 10, 100]:
  regressao_log = LogisticRegression(max_iter=1000, C=i)
  regressao_log.fit(X_treino_emb, y_treino)

  # Validação
  val_acc = regressao_log.score(X_validacao_emb, y_validacao)

  print(f"Para c = {i}    Val Accuracy: {val_acc:.4f}\n")



Para c = 0.001    Val Accuracy: 0.8841

Para c = 0.01    Val Accuracy: 0.9033

Para c = 0.1    Val Accuracy: 0.9101

Para c = 1    Val Accuracy: 0.9125

Para c = 10    Val Accuracy: 0.9119

Para c = 100    Val Accuracy: 0.9121



In [55]:
# Avaliação do Teste
regressao_log_melhor_c = LogisticRegression(max_iter=1000, C=1)
regressao_log_melhor_c.fit(X_treino_emb,y_treino)
y_pred = regressao_log_melhor_c.predict(X_teste_emb)

print("\n Matriz de Confusão")
print(confusion_matrix(y_teste, y_pred))

print("\n Classification Report")
print(classification_report(y_teste, y_pred, target_names=["Nao", "Sim"]))


 Matriz de Confusão
[[ 5950  1240]
 [ 1148 18119]]

 Classification Report
              precision    recall  f1-score   support

         Nao       0.84      0.83      0.83      7190
         Sim       0.94      0.94      0.94     19267

    accuracy                           0.91     26457
   macro avg       0.89      0.88      0.89     26457
weighted avg       0.91      0.91      0.91     26457



In [53]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs("/content/drive/MyDrive/embeddings/DistilBERT", exist_ok=True)

np.save("/content/drive/MyDrive/embeddings/DistilBERT/X_treino_emb_128.npy", X_treino_emb)
np.save("/content/drive/MyDrive/embeddings/DistilBERT/X_validacao_emb_128.npy", X_validacao_emb)
np.save("/content/drive/MyDrive/embeddings/DistilBERT/X_teste_emb_128.npy", X_teste_emb)
np.save("/content/drive/MyDrive/embeddings/DistilBERT/y_pred_distilbert_128.npy", y_pred)
np.save("/content/drive/MyDrive/embeddings/DistilBERT/y_teste_128.npy", y_teste.to_numpy())
np.save("/content/drive/MyDrive/embeddings/DistilBERT/X_teste_128.npy", X_teste.to_numpy())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
